# 01. Integración de datos

Se unen las exportaciones existentes de Telegram y Facebook. El cuaderno **no vuelve a realizar web scraping**, no limpia textos y no asigna polaridad.

La tabla principal tendrá únicamente:

| Columna | Significado |
|---|---|
| `id` | número sencillo y permanente dentro de esta versión: 1, 2, 3… |
| `fuente` | `telegram` o `facebook` |
| `texto_real` | texto exactamente como aparece en la exportación |

La procedencia de los archivos y sus huellas se conserva en un JSON separado, no como columnas repetidas en las 2.669 filas.


La reproducción lee las exportaciones guardadas y escribe en `salidas/<ejecución>/TABLAS`.

Si abres este cuaderno directamente en Colab, ejecuta su primera celda y autoriza Google Drive cuando se solicite. Para recorrer toda la secuencia de una vez, usa `INICIAR_EN_COLAB.ipynb`.

## A. Ubicar el proyecto

- **Qué hace:** define la carpeta `COLAB`.
- **Para qué:** permite usar las mismas rutas relativas en la computadora y en Google Colab.
- **Por qué:** evita escribir rutas personales dentro de cada lectura o exportación.


In [ ]:
# Inicio al abrir este cuaderno directamente en Google Colab.
from pathlib import Path

if not Path("corrida.json").is_file():
    import os
    import subprocess
    import sys

    try:
        from google.colab import drive
    except ImportError as error:
        raise FileNotFoundError("En Jupyter, abre la carpeta que contiene corrida.json.") from error

    proyecto_drive = Path("/content/drive/MyDrive/Tesis_RoBERTuito_M0_M1_M2")
    if not (proyecto_drive / "corrida.json").is_file():
        drive.mount("/content/drive", timeout_ms=180000)
    if not (proyecto_drive / "corrida.json").is_file():
        raise FileNotFoundError("Busca Tesis_RoBERTuito_M0_M1_M2 directamente en Mi unidad.")

    os.chdir(proyecto_drive)
    sys.path.insert(0, str(proyecto_drive))
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r",
         str(proyecto_drive / "requirements-cuadernos.txt")],
        check=True,
    )
    print("Carpeta de la tesis conectada:", proyecto_drive)

RAIZ = Path.cwd()
if not (RAIZ / "corrida.json").is_file():
    raise FileNotFoundError("Abre Jupyter en la carpeta que contiene corrida.json.")
print("Carpeta de trabajo:", RAIZ)


## B. Leer las dos exportaciones

- **Qué hace:** abre el archivo de Telegram y el archivo producido por Apify para Facebook.
- **Para qué:** recuperar únicamente el texto de cada publicación.
- **Por qué:** la clasificación de polaridad necesita texto y fuente; las demás columnas crudas no se repiten en el corpus de trabajo.


In [ ]:
import hashlib
import json
import pandas as pd
from IPython.display import display

RUTA_TELEGRAM = RAIZ / "data/raw/FINAL_Test_with_01670.xlsx"
RUTA_FACEBOOK = RAIZ / "data/raw/facebook_apify_2026-10-16_21-40-15-337.xlsx"

telegram = pd.read_excel(RUTA_TELEGRAM, sheet_name="Sheet1", keep_default_na=False)
facebook = pd.read_excel(RUTA_FACEBOOK, sheet_name="Data", keep_default_na=False)

if "Content" not in telegram or "text" not in facebook:
    raise ValueError("Las columnas de texto de las exportaciones cambiaron.")

display(pd.DataFrame({
    "fuente": ["telegram", "facebook"],
    "filas": [len(telegram), len(facebook)],
    "columna_usada": ["Content", "text"],
}))


## C. Unir y asignar el identificador

- **Qué hace:** coloca primero Telegram, después Facebook y asigna números consecutivos.
- **Para qué:** mantener un mismo `id` desde la integración hasta la evaluación.
- **Por qué:** un identificador corto facilita revisar una fila sin introducir hashes largos ni columnas auxiliares.


In [ ]:
datos = pd.concat([
    pd.DataFrame({"fuente": "telegram", "texto_real": telegram["Content"].astype(str)}),
    pd.DataFrame({"fuente": "facebook", "texto_real": facebook["text"].astype(str)}),
], ignore_index=True)
datos.insert(0, "id", range(1, len(datos) + 1))

if not datos["id"].is_unique:
    raise ValueError("El identificador debe ser único.")
if list(datos.columns) != ["id", "fuente", "texto_real"]:
    raise ValueError("La tabla principal debe conservar exactamente tres columnas.")

print("Filas integradas:", len(datos))
display(datos.head(10))


## D. Guardar el corpus y el registro técnico

- **Qué hace:** guarda la tabla mínima en Excel y un resumen técnico en JSON.
- **Para qué:** el Excel alimenta al cuaderno 02; el JSON demuestra qué archivos se utilizaron.
- **Por qué:** la trazabilidad es necesaria, pero no debe convertirse en columnas que el modelo nunca utilizará.


In [ ]:
from rutas import carpeta_salidas

def sha256(ruta):
    huella = hashlib.sha256()
    with ruta.open("rb") as archivo:
        for bloque in iter(lambda: archivo.read(1024 * 1024), b""):
            huella.update(bloque)
    return huella.hexdigest()

CARPETA = carpeta_salidas(RAIZ) / "TABLAS/01_integracion"
CARPETA.mkdir(parents=True, exist_ok=True)
SALIDA = CARPETA / "datos_integrados_v4.xlsx"
REGISTRO = CARPETA / "registro_fuentes_v4.json"

datos.to_excel(SALIDA, sheet_name="datos", index=False)
registro = {
    "version": "v4",
    "filas": int(len(datos)),
    "orden": "Telegram seguido de Facebook; id consecutivo desde 1",
    "fuentes": [
        {"fuente": "telegram", "archivo": RUTA_TELEGRAM.name,
         "hoja": "Sheet1", "filas": int(len(telegram)), "sha256": sha256(RUTA_TELEGRAM)},
        {"fuente": "facebook", "archivo": RUTA_FACEBOOK.name,
         "hoja": "Data", "filas": int(len(facebook)), "sha256": sha256(RUTA_FACEBOOK)},
    ],
}
REGISTRO.write_text(json.dumps(registro, ensure_ascii=False, indent=2), encoding="utf-8")

comprobacion = pd.read_excel(SALIDA, sheet_name="datos", keep_default_na=False)
if not comprobacion.equals(datos):
    raise ValueError("El archivo guardado no coincide con la tabla integrada.")
print("Guardado:", SALIDA)
print("Columnas finales:", list(comprobacion.columns))
display(comprobacion.head(10))
